In [12]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext
from torchtext.data import Field, TabularDataset, BucketIterator


batch_size = 32
learning_rate = 0.001
num_epochs = 10


TEXT = Field(sequential=True, tokenize='spacy', lower=True)
SENTIMENT = Field(sequential=False, use_vocab=False, dtype=torch.float)
df = pd.read_csv('./datasets/sentiment.csv')
fields = [('text', TEXT), ('sentiment', SENTIMENT)]
dataset = TabularDataset(
    path='./datasets/sentiment.csv',
    format='csv',
    fields=fields,
    skip_header=True
)


train_data, test_data = dataset.split(split_ratio=0.8)


TEXT.build_vocab(train_data, min_freq=5)


train_iterator, test_iterator = BucketIterator.splits(
    (train_data, test_data),
    batch_size=batch_size,
    sort_key=lambda x: len(x.text),
    sort_within_batch=True
)


class SentimentRegressor(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.fc = nn.Linear(embedding_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        hidden = torch.mean(embedded, dim=1)  
        hidden = torch.relu(self.fc(hidden))
        output = self.out(hidden)
        return output


input_dim = len(TEXT.vocab)
embedding_dim = 100
hidden_dim = 256
output_dim = 1

model = SentimentRegressor(input_dim, embedding_dim, hidden_dim, output_dim)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()


model.train()
for epoch in range(num_epochs):
    for batch in train_iterator:
        text = batch.text
        sentiment = batch.sentiment.view(-1, 1)
        optimizer.zero_grad()
        predictions = model(text).squeeze(1)
        loss = criterion(predictions, sentiment)
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch + 1}/{num_epochs}] | Loss: {loss.item():.4f}')


model.eval()
with torch.no_grad():
    total_loss = 0
    for batch in test_iterator:
        text = batch.text
        sentiment = batch.sentiment.view(-1, 1)
        predictions = model(text).squeeze(1)
        loss = criterion(predictions, sentiment)
        total_loss += loss.item()

    mean_loss = total_loss / len(test_iterator)
    print(f'Test Loss: {mean_loss:.4f}')


ImportError: cannot import name 'Field' from 'torchtext.data' (/Users/teddyoweh/Library/Python/3.9/lib/python/site-packages/torchtext/data/__init__.py)